# PSO Experimentos - Optimización de Pesos para Riesgo Cardiovascular

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from app.services.optimizers.pso import PSOOptimizer
from app.services.metrics_service import MetricsService

%matplotlib inline

## 1. Cargar datos sintéticos

In [ ]:
df = pd.read_csv('../app/data/synthetic_cases.csv')
print(f'Registros: {len(df)}')
print(df['risk_level'].value_counts())
df.head()

In [ ]:
def normalize_dataframe(df):
    hr_n = (df['heart_rate'] - 30) / 190
    spo2_n = (df['spo2'] - 60) / 40
    sbp_n = (df['systolic_bp'] - 60) / 200
    dbp_n = (df['diastolic_bp'] - 30) / 130
    chol_n = (df['cholesterol'] - 100) / 300
    glu_n = (df['glucose'] - 50) / 300
    age_n = df['age'] / 120
    chest_map = {'typical_angina': 0.25, 'atypical_angina': 0.50, 'non_anginal': 0.75, 'asymptomatic': 1.0}
    chest_n = df['chest_pain_type'].map(chest_map).fillna(0)
    sex_n = (df['sex'] == 'M').astype(float)
    features = pd.DataFrame({
        'heart_rate': hr_n, 'spo2': spo2_n, 'systolic_bp': sbp_n,
        'diastolic_bp': dbp_n, 'cholesterol': chol_n, 'glucose': glu_n,
        'age': age_n, 'chest_pain_type': chest_n, 'sex': sex_n,
        'smoker': df['smoker'].astype(float),
        'previous_condition': df['previous_condition'].astype(float),
    })
    return features.values, df['label'].values

X, y = normalize_dataframe(df)
print(f'Features shape: {X.shape}')
print(f'Distribución clases: {np.bincount(y)}')

## 2. Baseline (pesos uniformes)

In [ ]:
metrics = MetricsService()
baseline_w = np.ones(11) / 11.0
baseline_t = [0.40, 0.70]
baseline_results = metrics.evaluate(X, y, baseline_w, baseline_t)
print('Baseline:', baseline_results)

In [ ]:
scores_base = 1.0 / (1.0 + np.exp(-np.clip(X @ baseline_w, -20, 20)))
preds_base = np.where(scores_base < 0.40, 0, np.where(scores_base < 0.70, 1, 2))
cm_base = confusion_matrix(y, preds_base, labels=[0, 1, 2])
disp_base = ConfusionMatrixDisplay(cm_base, display_labels=['bajo', 'medio', 'alto'])
disp_base.plot(cmap='Blues')
plt.title('Matriz de Confusión - Baseline')
plt.savefig('../app/data/confusion_matrix_baseline.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Ejecutar PSO

In [ ]:
pso = PSOOptimizer(n_particles=30, max_iter=100, w=0.7, c1=1.5, c2=1.5)
result = pso.optimize(X, y)
print(f'Algoritmo: {result.algorithm}')
print(f'Generaciones: {result.n_generations}')
print(f'Tiempo: {result.computation_time}s')
print(f'Fitness final: {result.fitness_history[-1]:.4f}')
print(f'Weights: {[round(w, 3) for w in result.weights]}')
print(f'Thresholds: {[round(t, 3) for t in result.thresholds]}')
print(f'Bias: {result.bias:.3f}')

## 4. Curva de convergencia

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(result.fitness_history, color='#2c3e50', linewidth=1.5)
plt.xlabel('Generación')
plt.ylabel('Fitness')
plt.title('Curva de Convergencia - PSO')
plt.grid(alpha=0.3)
plt.savefig('../app/data/convergence_curve.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Evaluar modelo optimizado

In [ ]:
opt_w = np.array(result.weights)
opt_t = result.thresholds
opt_b = result.bias
optimized_results = metrics.evaluate(X, y, opt_w, opt_t, opt_b)
print('Optimizado:', optimized_results)

In [ ]:
scores_opt = 1.0 / (1.0 + np.exp(-np.clip(X @ opt_w + opt_b, -20, 20)))
preds_opt = np.where(scores_opt < opt_t[0], 0, np.where(scores_opt < opt_t[1], 1, 2))
cm_opt = confusion_matrix(y, preds_opt, labels=[0, 1, 2])
disp_opt = ConfusionMatrixDisplay(cm_opt, display_labels=['bajo', 'medio', 'alto'])
disp_opt.plot(cmap='Greens')
plt.title('Matriz de Confusión - Optimizado PSO')
plt.savefig('../app/data/confusion_matrix_optimized.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Comparación Baseline vs Optimizado

In [ ]:
comparison = metrics.compare(X, y, baseline_w, baseline_t, opt_w, opt_t, 0.0, opt_b)
comparison

## 7. Exportar pesos optimizados

In [ ]:
output = {
    "version": "2026-06-demo-v1",
    "algorithm": "PSO",
    "weights": result.weights,
    "thresholds": result.thresholds,
    "bias": result.bias,
    "fitness": result.fitness_history[-1],
    "metrics": comparison,
    "date": "2026-06-08",
    "parameters": {"n_particles": 30, "max_iter": 100, "w": 0.7, "c1": 1.5, "c2": 1.5},
    "description": "Pesos optimizados con PSO para modelo de riesgo cardiovascular"
}
with open('../app/data/optimized_weights.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Pesos exportados a app/data/optimized_weights.json')